# NACC_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'NACC'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = str(raw_primekg_path())

pd.set_option('display.max_rows', None)  # 行数无上限
pd.set_option('display.max_colwidth', None) # 单元格宽度无上限

def batch_inspect_ontology_nacc():
    print("==================================================")
    print("🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具 (NACC专版)...")
    
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        print(f"   ✅ 成功加载 {RAW_KG_PATH}。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧹 步骤 2: 划定安全检索范围...")
    target_types = ['disease', 'effect/phenotype']
    search_df = df[df['x_type'].isin(target_types)]
    print(f"   已将检索范围锁定在 {len(search_df)} 条 'disease' 和 'effect/phenotype' 关系中。")
    
    search_strategy = {
        "核心-轻度认知障碍 (MCI)": ["mild cognitive impairment", "cognitive decline"],
        "认知-记忆/失语/执行 (Memory/Language)": ["memory impairment", "amnesia", "aphasia", "executive functioning"],
        "精神-创伤/双相/强迫 (PTSD/Bipolar/OCD)": ["post-traumatic", "bipolar", "obsessive-compulsive", "schizophrenia"],
        "病史-心衰/房颤/脑缺血 (CHF/AFIB/TIA)": ["heart failure", "atrial fibrillation", "ischemic attack", "transient ischemic"],
        "病史-脑外伤/癫痫 (TBI/Seizure)": ["traumatic brain injury", "seizure", "epilepsy"],
        "病史-代谢/内分泌 (Diabetes/Thyroid)": ["diabetes mellitus", "thyroid", "vitamin b12"]
    }

    print("\n==================================================")
    print("🔍 步骤 3: 开始按医学词根进行批量模糊检索...")
    
    for category, keywords in search_strategy.items():
        print(f"\n▶ 正在检索【{category}】...")
        print(f"   使用的词根: {keywords}")
        
        pattern = '|'.join(keywords)
        mask = search_df['x_name'].str.contains(pattern, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个候选节点，前 10 个最精准的候选如下：")
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 使用这些词根未能找到任何节点。")

    print("\n==================================================")
    print("💡 步骤 4: 检修结果输出完毕。")
    print("==================================================")

if __name__ == "__main__":
    batch_inspect_ontology_nacc()